# R04 — protocol freeze (metadata only)

**Question:** Which exact replay, fit, calibration, split and evaluation rules will R05 execute? This notebook shows the sealed design, not empirical comparison results. Mini_7 is one development scene; C0–C3 and H are represented only by published archive metadata. All arithmetic examples below are labeled tests.

**Boundary:** no comparison/final cloud or label body is loaded. The operational scorer receives causal PCD/pose inputs only in later runs. Evaluator windows and labels join only after every fold prediction is sealed.

In [ ]:
import json, os, pathlib, csv, sys, html
from IPython.display import display, Markdown, HTML
sys.path.insert(0, str(pathlib.Path('src').resolve()))
run = pathlib.Path(os.environ['VP_RUN_DIR'])
roles = json.loads((pathlib.Path('configs')/'segment_roles.json').read_text())
method = json.loads((pathlib.Path('configs')/'replay_evaluation.json').read_text())
exposure = json.loads((run/'exposure_ledger.json').read_text())
def show_table(rows, columns=None):
    columns = columns or list(rows[0])
    display(HTML('<table><thead><tr>'+''.join('<th>'+html.escape(str(k))+'</th>' for k in columns)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+html.escape(str(row.get(k,'')))+'</td>' for k in columns)+'</tr>' for row in rows)+'</tbody></table>'))
def load_csv(name):
    with (run/(name+'.csv')).open(newline='') as f: return list(csv.DictReader(f))
display(Markdown('**Frozen content SHA-256:** `'+exposure['freeze_sha256']+'`'))
display(Markdown('**Input geometry:** native finite XYZ in metres; D radial crop [0,80) m with 4 ranges × 8 azimuth sectors = 32 cells; G 0.5 m voxels, directed q90 distance in metres. **Clock:** source nanoseconds, ideal arrival at source, latest past pose ≤150 ms, score deadline at event time.'))
show_table([{'role':x['id'],'segment':x['segment'],'published_bytes':x['bytes'],'published_lfs_sha256':x['published_lfs_sha256']} for x in [roles['development'],*roles['comparison'],roles['final']]])
show_table([exposure])

In [ ]:
splits = load_csv('T_R04_splits')
parameters = load_csv('T_R04_parameters')
lineage = load_csv('T_R04_lineage')
attack = load_csv('T_R04_attack_grid')
display(Markdown('### T_R04_splits — exact grouped roles'))
show_table(splits)
display(Markdown('### T_R04_parameters — every frozen parameter section'))
show_table(parameters)
display(Markdown('### T_R04_lineage — all 150 sensor × arm × version × fold keys'))
show_table(lineage)
display(Markdown('### Frozen intervention grid (no results)'))
show_table(attack)
assert len(lineage)==5*5*3*2 and len(attack)==30

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
fig, ax = plt.subplots(figsize=(9,3.5))
role_names=['C0','C1','C2','C3','H']
color={'reference':'#4d83b5','calibration':'#e2a345','test':'#b45a5a'}
for i, fold in enumerate(roles['folds']):
    for j, role in enumerate(role_names):
        kind='test' if role==fold['test'] else 'calibration' if role==fold['calibration'] else 'reference' if role in fold['reference'] else None
        if kind:
            ax.add_patch(Rectangle((j,i),1,1,facecolor=color[kind],edgecolor='white'))
            ax.text(j+.5,i+.5,kind[0].upper(),ha='center',va='center',color='white',weight='bold')
ax.set(xlim=(0,5),ylim=(5,0),xticks=[x+.5 for x in range(5)],xticklabels=role_names,yticks=[x+.5 for x in range(5)],yticklabels=[x['id'] for x in roles['folds']],xlabel='Published segment role',ylabel='Frozen fold',title='R04 split topology (metadata only)')
ax.tick_params(length=0)
fig.tight_layout()
for ext in ('png','svg','pdf'): fig.savefig(run/('F_R04_split_topology.'+ext),dpi=180)
display(fig)
plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(9,2.7))
windows=method['replay']['schedules_s']
for y,(name,intervals) in enumerate(windows.items()):
    for a,b in intervals: ax.broken_barh([(a,b-a)],(y-.28,.56),facecolors='#ae5d50')
ax.axvspan(0,3,color='#d9e9f1',alpha=.8,label='sender-only target prefix')
ax.set(xlim=(0,25),ylim=(-.6,1.6),yticks=[0,1],yticklabels=list(windows),xlabel='Acquisition elapsed time (s)',ylabel='Declared schedule',title='R04 fixed attempted windows; no observed alarms')
ax.legend(loc='upper right',fontsize=8)
fig.tight_layout()
for ext in ('png','svg','pdf'): fig.savefig(run/('F_R04_schedule.'+ext),dpi=180)
display(fig)
plt.close(fig)
display(Markdown('**Unknowns/failures kept:** G may be unsupported or exceed its pair budget; references may have zero scale; calibration can be insufficient or have no alarm power; reader rejection and zero-realized injections remain attempts. R05/R06 acquisition is resource-blocked until authorization and an actual retention budget are recorded.'))

In [ ]:
from valid_point.evaluation.metrics import event_metrics, episode_metrics, grouped_summary
fixture=[{'segment':'test_segment','episode':'fixture','principal':'003','sensor':'003','cloud_id':str(i),'alarm':a,'score':None if a is None else .6,'reader_valid':v,'realized_changed_rows':c,'injection_failed':not v} for i,a,v,c in [(1,True,True,1),(2,None,False,0),(3,False,True,1)]]
display(Markdown('### Arithmetic test fixture only — never empirical calibration'))
show_table(fixture)
show_table([event_metrics(fixture)])
assert event_metrics(fixture)['all_attempt_recall']['value']==1/3
assert event_metrics(fixture)['eligible_recall']['value']==1/2
assert event_metrics(fixture)['realized_recall']['value']==1/2
display(Markdown('**Gate:** split and metric tests plus a fresh top-to-bottom notebook establish technical integrity. Scientific detection/FPR/recovery outcomes remain uncomputed.'))